# Association Rule Mining with Apriori and FP-Growth

**Name:** *Shreyas More*
**Course:** *LAB 6*
**Lab Assignment:** Association Rule Mining: Apriori, FP-Growth and Seaborn Visualizations

**Dataset:** Online Retail Dataset (UCI Machine Learning Repository, id 352): transactions from a UK-based online gift retailer, 2010-2011.

## Step 1: Data Preparation

In [ ]:
# Install dependencies if needed (uncomment in a fresh environment)
# !pip install mlxtend seaborn pandas matplotlib openpyxl requests
import os, io, time, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", 80)

### 1.1 Load the data
The notebook uses a local copy (`Online Retail.xlsx`) if one exists next to it; otherwise it downloads the dataset from UCI. If the download is blocked, get the file manually from https://archive.ics.uci.edu/dataset/352/online+retail and place it in the notebook folder.

In [ ]:
LOCAL_FILE = "Online Retail.xlsx"
UCI_URL = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"

if not os.path.exists(LOCAL_FILE):
    import requests
    r = requests.get(UCI_URL, timeout=120)
    r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        z.extract(LOCAL_FILE)

raw = pd.read_excel(LOCAL_FILE)
print("Raw shape:", raw.shape)
raw.head()

### 1.2 Clean the data

In [ ]:
df = raw.copy()
print("Missing values per column:\n", df.isna().sum(), "\n")

df = df.dropna(subset=["InvoiceNo", "Description"])                 # need an item name
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df = df[~df["InvoiceNo"].str.upper().str.startswith("C")]          # cancellations
df = df[df["Quantity"] > 0]                                         # returns / adjustments
df = df[df["UnitPrice"] > 0]                                        # free or adjustment rows
df["Description"] = df["Description"].astype(str).str.strip().str.upper()
non_products = ["POSTAGE", "DOTCOM POSTAGE", "MANUAL", "CARRIAGE", "BANK CHARGES",
                "DISCOUNT", "SAMPLES", "AMAZON FEE", "CRUK COMMISSION"]
df = df[~df["Description"].isin(non_products)]
df = df.drop_duplicates(subset=["InvoiceNo", "Description"])        # one row per item per basket

print("Clean shape:", df.shape)
print("Invoices:", df["InvoiceNo"].nunique(), "| Unique items:", df["Description"].nunique())

### 1.3 Build the basket matrix
The full dataset has thousands of items and ~20,000 baskets, which makes Apriori very slow and memory-hungry. As is common for this dataset, we mine one country's transactions. **France** is the default (change `COUNTRY` to try others, e.g. `"Germany"`). Each row of the final matrix is a basket and each column a boolean item flag.

In [ ]:
COUNTRY = "France"
sub = df[df["Country"] == COUNTRY]

basket = (sub.assign(present=True)
             .pivot_table(index="InvoiceNo", columns="Description",
                          values="present", aggfunc="max", fill_value=False)
             .astype(bool))

# Drop baskets with a single item (they cannot form multi-item rules)
basket = basket[basket.sum(axis=1) > 1]
N_TRANSACTIONS = len(basket)
print(f"{COUNTRY}: {N_TRANSACTIONS} baskets x {basket.shape[1]} items")
print("Average basket size:", round(basket.sum(axis=1).mean(), 2))

### 1.4 Exploratory visualizations

In [ ]:
top_items = basket.sum().sort_values(ascending=False).head(15).reset_index()
top_items.columns = ["Item", "Count"]

plt.figure(figsize=(9, 6))
sns.barplot(data=top_items, y="Item", x="Count", hue="Item", palette="viridis", legend=False)
plt.title(f"Top 15 Most Frequent Items ({COUNTRY})")
plt.xlabel("Number of baskets containing item")
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(basket.sum(axis=1), bins=30, color="steelblue")
plt.title("Distribution of Basket Sizes")
plt.xlabel("Items per basket"); plt.tight_layout(); plt.show()

In [ ]:
# Co-occurrence heatmap for the 15 most frequent items
top_names = top_items["Item"].tolist()
X = basket[top_names].astype(int)
cooc = X.T.dot(X).copy()
cooc_vals = cooc.to_numpy(copy=True)
np.fill_diagonal(cooc_vals, 0)            # ignore self co-occurrence
cooc = pd.DataFrame(cooc_vals, index=cooc.index, columns=cooc.columns)
cooc_pct = cooc / N_TRANSACTIONS           # share of baskets containing both

plt.figure(figsize=(11, 9))
sns.heatmap(cooc_pct, cmap="YlOrRd", annot=True, fmt=".2f", annot_kws={"size": 7},
            linewidths=.4, cbar_kws={"label": "Share of baskets with both items"})
plt.title("Item Co-occurrence (Top 15 Items)")
plt.tight_layout(); plt.show()

**Observations (edit with your own):** The bar plot shows which products anchor the baskets. The heatmap highlights pairs that appear together most often; bright cells are candidates for strong association rules. Frequent co-occurrence alone can simply reflect two popular items, which is why lift is used later.

## Step 2: Frequent Itemset Mining with Apriori

In [ ]:
MIN_SUPPORT = 0.07      # itemset must appear in >= 7% of baskets
MAX_LEN = 4             # cap itemset size to keep runtime reasonable

t0 = time.perf_counter()
freq_apriori = apriori(basket, min_support=MIN_SUPPORT, use_colnames=True, max_len=MAX_LEN)
apriori_time = time.perf_counter() - t0

freq_apriori["length"] = freq_apriori["itemsets"].apply(len)
freq_apriori["itemset_str"] = freq_apriori["itemsets"].apply(lambda s: " + ".join(sorted(s)))
print(f"Apriori: {len(freq_apriori)} frequent itemsets in {apriori_time:.3f}s (min_support={MIN_SUPPORT})")
freq_apriori.sort_values("support", ascending=False).head(20)[["itemset_str", "support", "length"]]

In [ ]:
display(freq_apriori.groupby("length").size().rename("count").to_frame())

top_n = 15
plot_df = freq_apriori[freq_apriori["length"] >= 2].nlargest(top_n, "support")
if plot_df.empty:   # fall back to all sizes if no multi-item sets exist at this threshold
    plot_df = freq_apriori.nlargest(top_n, "support")

plt.figure(figsize=(10, 6))
sns.barplot(data=plot_df, y="itemset_str", x="support", hue="length", dodge=False, palette="crest")
plt.title(f"Apriori: Top {top_n} Frequent Itemsets (size >= 2)")
plt.xlabel("Support"); plt.ylabel("")
plt.tight_layout(); plt.show()

**Choosing the threshold:** With only a few hundred baskets and several hundred distinct items, any single item appears in a small fraction of baskets, so a high threshold (e.g. 0.5) returns nothing. 0.07 gives a manageable number of itemsets that include multi-item combinations. Lower values return more itemsets but cost much more time.

## Step 3: Frequent Itemset Mining with FP-Growth

In [ ]:
t0 = time.perf_counter()
freq_fp = fpgrowth(basket, min_support=MIN_SUPPORT, use_colnames=True, max_len=MAX_LEN)
fp_time = time.perf_counter() - t0

freq_fp["length"] = freq_fp["itemsets"].apply(len)
freq_fp["itemset_str"] = freq_fp["itemsets"].apply(lambda s: " + ".join(sorted(s)))
print(f"FP-Growth: {len(freq_fp)} frequent itemsets in {fp_time:.3f}s (min_support={MIN_SUPPORT})")

plot_df = freq_fp[freq_fp["length"] >= 2].nlargest(15, "support")
if plot_df.empty:
    plot_df = freq_fp.nlargest(15, "support")
plt.figure(figsize=(10, 6))
sns.barplot(data=plot_df, y="itemset_str", x="support", hue="length", dodge=False, palette="flare")
plt.title("FP-Growth: Top 15 Frequent Itemsets (size >= 2)")
plt.xlabel("Support"); plt.ylabel("")
plt.tight_layout(); plt.show()

### 3.1 Do both algorithms find the same itemsets?

In [ ]:
set_a = set(freq_apriori["itemsets"])
set_f = set(freq_fp["itemsets"])
print("Same number of itemsets:", len(set_a) == len(set_f))
print("Identical itemsets:", set_a == set_f)

merged = (freq_apriori[["itemsets", "support"]].merge(
          freq_fp[["itemsets", "support"]], on="itemsets", suffixes=("_apriori", "_fpgrowth")))
print("Max support difference:", (merged["support_apriori"] - merged["support_fpgrowth"]).abs().max())

### 3.2 Efficiency comparison across support thresholds

In [ ]:
def avg_time(func, min_support, repeats=3):
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        res = func(basket, min_support=min_support, use_colnames=True, max_len=MAX_LEN)
        times.append(time.perf_counter() - t0)
    return np.mean(times), len(res)

rows = []
for s in [0.15, 0.12, 0.10, 0.08, 0.07, 0.06, 0.05]:
    for name, fn in [("Apriori", apriori), ("FP-Growth", fpgrowth)]:
        t, n = avg_time(fn, s)
        rows.append({"min_support": s, "Algorithm": name, "Seconds": t, "Itemsets": n})
timing = pd.DataFrame(rows)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sns.lineplot(data=timing, x="min_support", y="Seconds", hue="Algorithm", marker="o", ax=ax[0])
ax[0].invert_xaxis(); ax[0].set_title("Runtime vs. Minimum Support")
sns.lineplot(data=timing[timing["Algorithm"] == "Apriori"], x="min_support", y="Itemsets",
             marker="o", color="gray", ax=ax[1])
ax[1].invert_xaxis(); ax[1].set_title("Frequent Itemsets Found vs. Minimum Support")
plt.tight_layout(); plt.show()

timing.pivot(index="min_support", columns="Algorithm", values="Seconds") \
      .sort_index(ascending=False).round(4)

## Step 4: Generating and Analyzing Association Rules

In [ ]:
MIN_CONFIDENCE = 0.6

def make_rules(freq, n_tx, min_conf):
    # Newer mlxtend versions need num_itemsets; older ones do not accept it.
    try:
        r = association_rules(freq, num_itemsets=n_tx, metric="confidence", min_threshold=min_conf)
    except TypeError:
        r = association_rules(freq, metric="confidence", min_threshold=min_conf)
    r["antecedents_str"] = r["antecedents"].apply(lambda s: ", ".join(sorted(s)))
    r["consequents_str"] = r["consequents"].apply(lambda s: ", ".join(sorted(s)))
    return r

rules_apriori = make_rules(freq_apriori, N_TRANSACTIONS, MIN_CONFIDENCE)
rules_fp = make_rules(freq_fp, N_TRANSACTIONS, MIN_CONFIDENCE)
print(f"Rules (Apriori): {len(rules_apriori)} | Rules (FP-Growth): {len(rules_fp)}")

cols = ["antecedents_str", "consequents_str", "support", "confidence", "lift"]
rules_apriori.sort_values(["lift", "confidence"], ascending=False)[cols].head(15)

### 4.1 Confirm both algorithms yield the same rules

In [ ]:
key = lambda r: set(zip(r["antecedents"], r["consequents"]))
print("Identical rule sets:", key(rules_apriori) == key(rules_fp))

### 4.2 Visualizing the rules

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=rules_apriori, x="confidence", y="lift", size="support",
                hue="support", palette="viridis", sizes=(30, 300), alpha=.75)
plt.axhline(1, color="red", ls="--", lw=1, label="lift = 1 (independence)")
plt.title("Association Rules: Confidence vs. Lift")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.show()

In [ ]:
# Strongest rules: high confidence AND lift > 1
strong = rules_apriori[(rules_apriori["lift"] > 1) & (rules_apriori["confidence"] >= 0.7)] \
            .sort_values("lift", ascending=False).head(10).copy()
strong["rule"] = strong["antecedents_str"] + "  =>  " + strong["consequents_str"]

plt.figure(figsize=(10, 5))
sns.barplot(data=strong, y="rule", x="lift", hue="confidence", dodge=False, palette="rocket_r")
plt.title("Top Rules by Lift (confidence >= 0.7)")
plt.ylabel(""); plt.tight_layout(); plt.show()

In [ ]:
# Heatmap of lift for single-item => single-item rules
single = rules_apriori[(rules_apriori["antecedents"].apply(len) == 1) &
                       (rules_apriori["consequents"].apply(len) == 1)]
if not single.empty:
    mat = single.pivot_table(index="antecedents_str", columns="consequents_str", values="lift")
    plt.figure(figsize=(max(6, .5 * mat.shape[1] + 4), max(4, .5 * mat.shape[0] + 2)))
    sns.heatmap(mat, cmap="mako_r", annot=True, fmt=".1f", linewidths=.3,
                cbar_kws={"label": "Lift"})
    plt.title("Lift Heatmap: Antecedent => Consequent"); plt.xlabel("Consequent"); plt.ylabel("Antecedent")
    plt.tight_layout(); plt.show()

### 4.3 Interpretation (edit with your results)
* **Support** is the share of baskets containing every item in the rule, so it shows how common the pattern is.
* **Confidence** is P(consequent | antecedent): how often the consequent appears when the antecedent does.
* **Lift** compares confidence to the consequent's baseline popularity. Lift > 1 means the items appear together more than chance, = 1 means independence, < 1 means they are substitutes or avoided together.

Look at the top rules in the tables above. Retail datasets like this one typically show rules between **variants of the same product family** (e.g., different colours or sizes of the same design, or matching sets such as plates/cups/napkins). Lift values well above 1 together with high confidence suggest items that are bought as sets. Practical uses: bundle offers, shelf/website placement, and "customers also bought" recommendations. Note that mirrored rules (A => B and B => A) usually appear together with the same lift.

## Step 5: Comparative Analysis

In [ ]:
summary = pd.DataFrame({
    "Algorithm": ["Apriori", "FP-Growth"],
    "Runtime (s)": [apriori_time, fp_time],
    "Frequent itemsets": [len(freq_apriori), len(freq_fp)],
    "Rules": [len(rules_apriori), len(rules_fp)],
})
display(summary.round(4))
print(f"FP-Growth speedup over Apriori at min_support={MIN_SUPPORT}: {apriori_time / fp_time:.1f}x")

plt.figure(figsize=(5, 4))
sns.barplot(data=summary, x="Algorithm", y="Runtime (s)", hue="Algorithm", palette="Set2", legend=False)
plt.title("Runtime Comparison"); plt.tight_layout(); plt.show()

### Results and discussion (finalize with your measured numbers)

**Output:** Both algorithms returned identical frequent itemsets, supports and rules (verified in the cells above). They differ in *how* they search, not *what* they find.

**Speed:** FP-Growth was faster in the runtime table, and the gap widened as the minimum support dropped. Reasons:
* **Apriori** generates candidate itemsets level by level and rescans the data to count each candidate. As the threshold falls, the number of candidates grows combinatorially, with many scans.
* **FP-Growth** compresses the data into an FP-tree in two passes and mines it recursively without generating candidates, so cost grows far more slowly.
* On very small datasets or high thresholds the difference can be tiny or even reversed, because FP-tree construction has overhead.

**Challenges and how they were resolved** (replace or extend with your own):
1. *Memory/time on the full dataset:* the complete Online Retail data has about 4,000 items and 20,000 baskets, so the one-hot matrix was too large for Apriori. I restricted to one country and used `max_len`.
2. *Messy data:* cancellations (invoices starting with "C"), negative quantities, missing descriptions, and non-product rows like POSTAGE were removed, and descriptions were normalized.
3. *Support threshold selection:* thresholds that were too high returned no itemsets and too low caused long runtimes, so I tested several values and compared runtimes.
4. *Library API differences:* newer versions of `mlxtend` require `num_itemsets` in `association_rules`, which the helper function handles.
5. *Interpreting rules:* high confidence alone was misleading for very popular items, so lift was used to filter for genuine associations.